# Production Trade Report

Pull historical trade data directly from the local MetaTrader 5 terminal and summarise production performance by symbol.

**Workflow**
1. Set the date range and optional symbol/group filters
2. Run the fetch cell to load MT5 deal history
3. Review the per-symbol summary, daily PnL, and best/worst trades


In [40]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

from Learn.report import fetch_trade_report

In [ ]:
# -- Configuration -----------------------------------------------------------
SYMBOL      = "XAUUSD"  # Base symbol name (must match data/{SYMBOL}_M1_520weeks.csv)

START_DATE  = "2026-04-27"
END_DATE    = "2026-05-02"

GROUP       = "*"      # MT5 group filter, e.g. "*USD*" or "*"
SYMBOLS     = None     # e.g. ["EURUSD.a", "US500.a"]
CLOSED_ONLY = True     # Summary focuses on close-side deals only


In [42]:
trades, summary = fetch_trade_report(
    start_date=START_DATE,
    end_date=END_DATE,
    group=GROUP,
    symbols=SYMBOLS,
    closed_only=CLOSED_ONLY,
)

closed_trades = trades[trades["entry_type"].isin(["OUT", "OUT_BY", "INOUT"])].copy()
closed_trades["trade_date"] = closed_trades["time"].dt.tz_convert(None).dt.floor("D") if not closed_trades.empty else pd.Series(dtype="datetime64[ns]")


In [43]:
print(f"Rows returned      : {len(trades):,}")
print(f"Closed-trade rows  : {len(closed_trades):,}")
print(f"Symbols returned   : {sorted(trades['symbol'].dropna().unique().tolist()) if not trades.empty else []}")
if not trades.empty:
    print(f"Time range (UTC)   : {trades['time'].min()} -> {trades['time'].max()}")

display(trades.head(10))

Rows returned      : 118
Closed-trade rows  : 59
Symbols returned   : ['EURUSD.a', 'US500.a', 'XAUUSD.a']
Time range (UTC)   : 2026-04-27 14:05:40+00:00 -> 2026-05-01 20:29:09+00:00


,ticket,order,position_id,time,time_msc,symbol,side,deal_type,entry_type,reason,volume,price,profit,commission,swap,fee,net_pnl,magic,comment,external_id
0,233561667,293663305,293663305,2026-04-27 14:05:40+00:00,2026-04-27 14:05:40.769000+00:00,XAUUSD.a,buy,BUY,IN,EXPERT,0.10,4706.40000,0.00,0.00,0.0,0.0,0.00,235001,pending_stop_ord,
1,233564378,293669381,293663305,2026-04-27 14:16:02+00:00,2026-04-27 14:16:02.003000+00:00,XAUUSD.a,sell,SELL,OUT,SL,0.10,4702.75000,-50.82,0.00,0.0,0.0,-50.82,235001,[sl 4702.75],
2,233592635,293720485,293720485,2026-04-27 15:25:21+00:00,2026-04-27 15:25:21.204000+00:00,EURUSD.a,buy,BUY,IN,EXPERT,1.25,1.17494,0.00,-4.38,0.0,0.0,-4.38,235000,pending_stop_ord,
3,233610277,293745684,293745684,2026-04-27 15:50:32+00:00,2026-04-27 15:50:32.621000+00:00,XAUUSD.a,sell,SELL,IN,EXPERT,0.10,4702.59000,0.00,0.00,0.0,0.0,0.00,235001,pending_stop_ord,
4,233615658,293753891,293720485,2026-04-27 15:58:37+00:00,2026-04-27 15:58:37.360000+00:00,EURUSD.a,sell,SELL,OUT,SL,1.25,1.17453,-71.30,-4.38,0.0,0.0,-75.68,235000,[sl 1.17453],
5,233618859,293758393,293745684,2026-04-27 16:01:31+00:00,2026-04-27 16:01:31.829000+00:00,XAUUSD.a,buy,BUY,OUT,TP,0.10,4696.66000,82.51,0.00,0.0,0.0,82.51,235001,[tp 4696.66],
6,233633597,293777907,293777907,2026-04-27 16:19:07+00:00,2026-04-27 16:19:07.334000+00:00,XAUUSD.a,sell,SELL,IN,EXPERT,0.10,4700.89000,0.00,0.00,0.0,0.0,0.00,235001,pending_stop_ord,
7,233643984,293794668,293777907,2026-04-27 16:33:41+00:00,2026-04-27 16:33:41.907000+00:00,XAUUSD.a,buy,BUY,OUT,TP,0.10,4694.83000,84.30,0.00,0.0,0.0,84.30,235001,[tp 4694.83],
8,233670082,293825512,293825512,2026-04-27 17:00:06+00:00,2026-04-27 17:00:06.095000+00:00,XAUUSD.a,sell,SELL,IN,EXPERT,0.10,4695.37000,0.00,0.00,0.0,0.0,0.00,235001,pending_stop_ord,
9,233675666,293832519,293832519,2026-04-27 17:07:04+00:00,2026-04-27 17:07:04.961000+00:00,EURUSD.a,sell,SELL,IN,EXPERT,1.25,1.17398,0.00,-4.38,0.0,0.0,-4.38,235000,pending_stop_ord,


## Per-symbol summary

In [44]:
if summary.empty:
    print("No symbol summary available for the selected range/filter.")
else:
    summary_display = summary.copy()
    summary_display["win_rate"] = (summary_display["win_rate"] * 100).round(2)
    display(summary_display.round({
        "volume_lots": 2,
        "gross_profit": 2,
        "gross_loss": 2,
        "net_pnl": 2,
        "avg_net_pnl": 2,
        "median_net_pnl": 2,
        "win_rate": 2,
        "avg_win": 2,
        "avg_loss": 2,
        "total_commission": 2,
        "total_swap": 2,
        "total_fee": 2,
    }))

,symbol,trade_count,first_trade_time,last_trade_time,volume_lots,gross_profit,gross_loss,net_pnl,avg_net_pnl,median_net_pnl,win_rate,avg_win,avg_loss,total_commission,total_swap,total_fee
0,XAUUSD.a,38,2026-04-27 14:16:02+00:00,2026-05-01 20:29:09+00:00,3.80,1883.50,-1292.56,590.94,15.55,65.45,57.89,85.61,-80.78,0.00,0.0,0.0
1,US500.a,10,2026-04-28 04:35:43+00:00,2026-05-01 18:37:39+00:00,38.80,205.31,-34.17,171.14,17.11,26.57,80.00,25.66,-17.08,0.00,0.0,0.0
2,EURUSD.a,11,2026-04-27 15:58:37+00:00,2026-05-01 17:36:04+00:00,13.75,242.05,-606.08,-364.03,-33.09,-75.68,36.36,60.51,-86.58,-48.18,0.0,0.0


## OHLCV Candlestick Chart

Load 1-minute OHLCV data for the selected symbol and display it as an interactive candlestick chart with volume. Trade entries and exits from the MT5 report are overlaid as markers.


In [45]:
from pathlib import Path

_ohlcv_path = Path("../data") / f"{SYMBOL}_M1_520weeks.csv"

if not _ohlcv_path.exists():
    print(f"OHLCV file not found: {_ohlcv_path}")
    ohlcv = None
else:
    ohlcv = pd.read_csv(
        _ohlcv_path,
        parse_dates=["Time"],
        index_col="Time",
    )
    ohlcv.index = pd.to_datetime(ohlcv.index, utc=True)
    _start = pd.Timestamp(START_DATE, tz="UTC")
    _end   = pd.Timestamp(END_DATE,   tz="UTC") + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
    ohlcv  = ohlcv.loc[_start:_end]
    n_bars = len(ohlcv)
    print(f"Loaded {n_bars:,} 1-minute bars for {SYMBOL} ({ohlcv.index.min()} -> {ohlcv.index.max()})")


Loaded 6,767 1-minute bars for XAUUSD (2026-04-27 00:00:00+00:00 -> 2026-05-01 20:54:00+00:00)


In [62]:
if ohlcv is None or ohlcv.empty:
    print("No OHLCV data available to chart.")
else:
    # --- Pair IN/OUT deals for the selected symbol ---
    _sym_trades = (
        trades[trades["symbol"].str.startswith(SYMBOL)].copy()
        if not trades.empty
        else pd.DataFrame()
    )

    _entries = (
        _sym_trades[_sym_trades["entry_type"] == "IN"][
            ["position_id", "time", "price", "side"]
        ].rename(columns={"time": "entry_time", "price": "entry_price"})
    )
    _exits = (
        _sym_trades[_sym_trades["entry_type"].isin(["OUT", "OUT_BY", "INOUT"])][
            ["position_id", "time", "price", "net_pnl"]
        ].rename(columns={"time": "exit_time", "price": "exit_price"})
    )

    _pairs = _entries.merge(_exits, on="position_id", how="inner").copy()
    _pairs["result"] = _pairs["net_pnl"].apply(
        lambda x: "win" if x > 0 else ("loss" if x < 0 else "breakeven")
    )

    # MT5 deal timestamps are broker server time (UTC+3). Subtract 3 hours to
    # convert to true UTC before flooring to the 1-min OHLCV bar boundary.
    _MT5_TZ_OFFSET = pd.Timedelta(hours=3)
    _pairs["entry_bar"] = (_pairs["entry_time"] - _MT5_TZ_OFFSET).dt.floor("min")
    _pairs["exit_bar"]  = (_pairs["exit_time"]  - _MT5_TZ_OFFSET).dt.floor("min")

    _result_color = {"win": "lime", "loss": "red", "breakeven": "grey"}
    _side_color   = {"buy": "#00cc44", "sell": "#ff3333"}
    _entry_symbol = {"buy": "triangle-up", "sell": "triangle-down"}
    _results = ["win", "loss", "breakeven"]
    _sides   = ["buy", "sell"]

    # Trading session definitions (UTC hours): name, open, close, fill color, legend color
    _sessions = [
        ("Asian",    0,  9,  "rgba(100, 180, 255, 0.12)", "#64b4ff"),
        ("European", 7,  16, "rgba(100, 210, 130, 0.12)", "#64d282"),
        ("US",       13, 22, "rgba(255, 160, 80,  0.12)", "#ffa050"),
    ]

    # --- Build chart ---
    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        row_heights=[0.75, 0.25],
        vertical_spacing=0.04,
        subplot_titles=(f"{SYMBOL} — Candlestick", "Volume"),
    )

    # Trace 0: Candlestick
    fig.add_trace(
        go.Candlestick(
            x=ohlcv.index,
            open=ohlcv["Open"],
            high=ohlcv["High"],
            low=ohlcv["Low"],
            close=ohlcv["Close"],
            name=SYMBOL,
            increasing_line_color="#26a69a",
            decreasing_line_color="#ef5350",
        ),
        row=1, col=1,
    )

    # Trace 1: Volume
    fig.add_trace(
        go.Bar(
            x=ohlcv.index,
            y=ohlcv["Volume"],
            name="Volume",
            marker_color="rgba(100,100,200,0.4)",
            showlegend=False,
        ),
        row=2, col=1,
    )

    # --- Session background shading (shapes — no effect on trace index or _vis()) ---
    _chart_dates = pd.date_range(
        start=ohlcv.index.normalize().min(),
        end=ohlcv.index.normalize().max(),
        freq="D",
        tz="UTC",
    )
    for _date in _chart_dates:
        for _sname, _h0, _h1, _fill, _ in _sessions:
            fig.add_vrect(
                x0=_date + pd.Timedelta(hours=_h0),
                x1=_date + pd.Timedelta(hours=_h1),
                fillcolor=_fill,
                layer="below",
                line_width=0,
                row=1, col=1,
            )

    # Traces 2-4: Session legend dummy traces (always visible, never toggled)
    for _sname, _h0, _h1, _fill, _legend_color in _sessions:
        fig.add_trace(
            go.Scatter(
                x=[None], y=[None],
                mode="markers",
                name=f"{_sname} session",
                marker=dict(symbol="square", size=12, color=_legend_color),
                showlegend=True,
            ),
            row=1, col=1,
        )

    # Traces 5-7: Connector lines, one trace per result using None-separated segments
    for result in _results:
        _r = _pairs[_pairs["result"] == result]
        x_segs, y_segs = [], []
        for _, row_data in _r.iterrows():
            x_segs.extend([row_data["entry_bar"], row_data["exit_bar"], None])
            y_segs.extend([row_data["entry_price"], row_data["exit_price"], None])
        fig.add_trace(
            go.Scatter(
                x=x_segs or [None],
                y=y_segs or [None],
                mode="lines",
                line=dict(color=_result_color[result], width=1, dash="dot"),
                showlegend=False,
                hoverinfo="skip",
                name=f"_connector_{result}",
            ),
            row=1, col=1,
        )

    # Traces 8-13: Entry markers, one trace per (side x result)
    for side in _sides:
        for result in _results:
            _s = _pairs[(_pairs["side"] == side) & (_pairs["result"] == result)]
            fig.add_trace(
                go.Scatter(
                    x=_s["entry_bar"].tolist() if not _s.empty else [],
                    y=_s["entry_price"].tolist() if not _s.empty else [],
                    mode="markers",
                    name=f"Entry {side} ({result})",
                    marker=dict(
                        symbol=_entry_symbol[side],
                        color=_result_color[result],
                        size=10,
                        line=dict(color="white", width=1),
                    ),
                    customdata=_s[["position_id", "entry_price", "entry_time"]].values if not _s.empty else [],
                    hovertemplate=(
                        "Entry %{customdata[0]}<br>"
                        "Price: %{customdata[1]:.5f}<br>"
                        "Time: %{customdata[2]}<extra></extra>"
                    ),
                ),
                row=1, col=1,
            )

    # Traces 14-16: Exit markers, one trace per result
    for result in _results:
        _r = _pairs[_pairs["result"] == result]
        fig.add_trace(
            go.Scatter(
                x=_r["exit_bar"].tolist() if not _r.empty else [],
                y=_r["exit_price"].tolist() if not _r.empty else [],
                mode="markers",
                name=f"Exit ({result})",
                marker=dict(
                    symbol="circle",
                    color=_result_color[result],
                    size=8,
                    line=dict(color="white", width=1),
                ),
                customdata=_r[["position_id", "exit_price", "net_pnl", "exit_time"]].values if not _r.empty else [],
                hovertemplate=(
                    "Exit %{customdata[0]}<br>"
                    "Price: %{customdata[1]:.5f}<br>"
                    "Net PnL: %{customdata[2]:.2f}<br>"
                    "Time: %{customdata[3]}<extra></extra>"
                ),
            ),
            row=1, col=1,
        )

    # Fixed trace layout (17 traces total):
    #   0: candlestick,  1: volume
    #   2-4: session legend dummies (Asian, European, US) — always visible
    #   5-7: connectors (win, loss, breakeven)
    #   8-13: entry markers (buy+win, buy+loss, buy+breakeven, sell+win, sell+loss, sell+breakeven)
    #   14-16: exit markers (win, loss, breakeven)
    def _vis(results_shown):
        show = set(results_shown)
        v = [True, True]        # candlestick + volume always visible
        v += [True, True, True] # session legend dummies always visible
        for r in _results:      # connectors
            v.append(r in show)
        for _ in _sides:        # entry markers (buy then sell, each with 3 results)
            for r in _results:
                v.append(r in show)
        for r in _results:      # exit markers
            v.append(r in show)
        return v

    fig.update_layout(
        title=f"{SYMBOL} — OHLCV with Trades ({START_DATE} to {END_DATE})",
        template="plotly_white",
        height=750,
        hovermode="x unified",
        xaxis_rangeslider_visible=False,
        updatemenus=[
            dict(
                type="buttons",
                direction="right",
                x=0.0,
                y=1.12,
                xanchor="left",
                yanchor="top",
                showactive=True,
                buttons=[
                    dict(
                        label="All trades",
                        method="restyle",
                        args=[{"visible": _vis(["win", "loss", "breakeven"])}],
                    ),
                    dict(
                        label="Wins only",
                        method="restyle",
                        args=[{"visible": _vis(["win"])}],
                    ),
                    dict(
                        label="Losses only",
                        method="restyle",
                        args=[{"visible": _vis(["loss"])}],
                    ),
                ],
            )
        ],
    )
    fig.update_yaxes(title_text="Price", row=1, col=1)
    fig.update_yaxes(title_text="Volume", row=2, col=1)
    fig.update_xaxes(title_text="Time (UTC)", row=2, col=1)
    fig.show()
    print(f"Trades overlaid: {len(_pairs)} matched entry/exit pairs")


Trades overlaid: 38 matched entry/exit pairs


## Daily net PnL and cumulative PnL

In [47]:
if closed_trades.empty:
    print("No closed trades available to chart.")
else:
    daily = (
        closed_trades.groupby(["trade_date", "symbol"], as_index=False)["net_pnl"]
        .sum()
        .sort_values(["trade_date", "symbol"])
    )
    daily_pivot = daily.pivot(index="trade_date", columns="symbol", values="net_pnl").fillna(0.0)
    cumulative = daily_pivot.cumsum()

    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.10,
        subplot_titles=("Daily net PnL by symbol", "Cumulative net PnL by symbol"),
    )

    for symbol in daily_pivot.columns:
        fig.add_trace(
            go.Scatter(
                x=daily_pivot.index,
                y=daily_pivot[symbol],
                mode="lines+markers",
                name=f"{symbol} daily",
                legendgroup=str(symbol),
            ),
            row=1,
            col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=cumulative.index,
                y=cumulative[symbol],
                mode="lines",
                name=f"{symbol} cumulative",
                legendgroup=str(symbol),
                showlegend=False,
            ),
            row=2,
            col=1,
        )

    zero_line_style = dict(color="black", width=1, dash="dash")
    fig.add_hline(y=0, line=zero_line_style, row=1, col=1)
    fig.add_hline(y=0, line=zero_line_style, row=2, col=1)
    fig.update_yaxes(title_text="Net PnL", row=1, col=1)
    fig.update_yaxes(title_text="Cumulative net PnL", row=2, col=1)
    fig.update_xaxes(title_text="Trade date", row=2, col=1)
    fig.update_layout(height=850, hovermode="x unified", template="plotly_white")
    fig.show()

    display(daily.tail(20))

,trade_date,symbol,net_pnl
0,2026-04-27,EURUSD.a,-163.43
1,2026-04-27,XAUUSD.a,369.61
2,2026-04-28,EURUSD.a,123.47
3,2026-04-28,US500.a,11.14
4,2026-04-28,XAUUSD.a,-133.08
5,2026-04-29,EURUSD.a,37.52
6,2026-04-29,US500.a,20.22
7,2026-04-29,XAUUSD.a,140.28
8,2026-04-30,EURUSD.a,-57.01
9,2026-04-30,US500.a,112.39


In [48]:
if closed_trades.empty:
    print("No closed trades available to chart.")
else:
    plot_df = closed_trades.copy()
    plot_df["result"] = plot_df["net_pnl"].apply(lambda x: "win" if x > 0 else ("loss" if x < 0 else "breakeven"))
    plot_df["abs_profit"] = plot_df["net_pnl"].abs()

    # Remove top 5% outliers per symbol
    # p95_by_symbol = plot_df.groupby("symbol")["abs_profit"].transform(lambda s: s.quantile(0.95))
    # plot_df = plot_df[plot_df["abs_profit"] < p95_by_symbol]

    symbols = sorted(plot_df["symbol"].dropna().unique().tolist())
    if not symbols:
        print("No data left after filtering.")
    else:
        n_cols = 2
        n_rows = (len(symbols) + n_cols - 1) // n_cols

        fig = make_subplots(
            rows=n_rows,
            cols=n_cols,
            subplot_titles=[f"{s}" for s in symbols],
            vertical_spacing=0.10,
            horizontal_spacing=0.08,
        )

        for i, sym in enumerate(symbols):
            r = i // n_cols + 1
            c = i % n_cols + 1

            symbol_df = plot_df[plot_df["symbol"] == sym]
            max_val = symbol_df["abs_profit"].max()
            bin_size = max_val / 30 if max_val > 0 else 1

            fig.add_trace(
                go.Histogram(
                    x=symbol_df[symbol_df["result"] == "win"]["abs_profit"],
                    name="Wins",
                    marker_color="green",
                    opacity=0.75,
                    xbins=dict(start=0, end=max_val, size=bin_size),
                    legendgroup="wins",
                    showlegend=(i == 0),
                ),
                row=r,
                col=c,
            )
            fig.add_trace(
                go.Histogram(
                    x=symbol_df[symbol_df["result"] == "loss"]["abs_profit"],
                    name="Losses",
                    marker_color="red",
                    opacity=0.75,
                    xbins=dict(start=0, end=max_val, size=bin_size),
                    legendgroup="losses",
                    showlegend=(i == 0),
                ),
                row=r,
                col=c,
            )

            fig.update_xaxes(title_text="Absolute Profit", row=r, col=c)
            fig.update_yaxes(title_text="Count", row=r, col=c)

        fig.update_layout(
            title="Distribution of absolute profit for closed trades by symbol",
            barmode="overlay",
            template="plotly_white",
            height=max(400, 320 * n_rows),
        )
        fig.show()

## Best and worst closed trades

In [49]:
if closed_trades.empty:
    print("No closed trades available for ranking.")
else:
    cols = ["time", "symbol", "side", "entry_type", "volume", "price", "profit", "commission", "swap", "fee", "net_pnl", "comment"]
    print("Top 10 winners")
    display(closed_trades.sort_values("net_pnl", ascending=False)[cols].head(10))
    print("Top 10 losers")
    display(closed_trades.sort_values("net_pnl", ascending=True)[cols].head(10))

Top 10 winners


,time,symbol,side,entry_type,volume,price,profit,commission,swap,fee,net_pnl,comment
80,2026-04-30 17:36:34+00:00,XAUUSD.a,buy,OUT,0.10,4613.12000,134.50,0.00,0.0,0.0,134.50,[tp 4613.12]
57,2026-04-29 18:36:11+00:00,XAUUSD.a,sell,OUT,0.10,4548.21000,111.83,0.00,0.0,0.0,111.83,[tp 4548.21]
15,2026-04-27 19:00:41+00:00,XAUUSD.a,buy,OUT,0.10,4668.09000,109.56,0.00,0.0,0.0,109.56,[tp 4668.09]
107,2026-05-01 16:33:57+00:00,XAUUSD.a,sell,OUT,0.10,4600.08000,107.52,0.00,0.0,0.0,107.52,[tp 4600.08]
61,2026-04-29 22:49:08+00:00,XAUUSD.a,sell,OUT,0.10,4546.25000,96.99,0.00,0.0,0.0,96.99,[tp 4546.25]
55,2026-04-29 18:00:40+00:00,XAUUSD.a,sell,OUT,0.10,4542.15000,96.05,0.00,0.0,0.0,96.05,[tp 4542.15]
10,2026-04-27 17:09:58+00:00,XAUUSD.a,buy,OUT,0.10,4688.72000,92.50,0.00,0.0,0.0,92.50,[tp 4688.72]
29,2026-04-28 18:26:15+00:00,EURUSD.a,sell,OUT,1.25,1.17104,95.85,-4.38,0.0,0.0,91.47,[tp 1.17104]
12,2026-04-27 17:27:15+00:00,XAUUSD.a,buy,OUT,0.10,4687.78000,89.69,0.00,0.0,0.0,89.69,[tp 4687.78]
115,2026-05-01 19:17:00+00:00,XAUUSD.a,sell,OUT,0.10,4648.46000,86.98,0.00,0.0,0.0,86.98,[tp 4648.46]


Top 10 losers


,time,symbol,side,entry_type,volume,price,profit,commission,swap,fee,net_pnl,comment
112,2026-05-01 17:50:06+00:00,XAUUSD.a,sell,OUT,0.10,4628.50000,-144.80,0.00,0.0,0.0,-144.80,[sl 4628.50]
27,2026-04-28 16:07:26+00:00,XAUUSD.a,sell,OUT,0.10,4578.25000,-137.37,0.00,0.0,0.0,-137.37,[sl 4578.25]
109,2026-05-01 17:36:04+00:00,EURUSD.a,sell,OUT,1.25,1.17725,-121.28,-4.38,0.0,0.0,-125.66,[sl 1.17725]
117,2026-05-01 20:29:09+00:00,XAUUSD.a,sell,OUT,0.10,4633.49000,-105.52,0.00,0.0,0.0,-105.52,[sl 4633.49]
23,2026-04-28 10:04:32+00:00,XAUUSD.a,buy,OUT,0.10,4636.75000,-103.53,0.00,0.0,0.0,-103.53,[sl 4636.75]
77,2026-04-30 16:30:35+00:00,XAUUSD.a,sell,OUT,0.10,4629.90000,-101.59,0.00,0.0,0.0,-101.59,[sl 4629.90]
106,2026-05-01 15:46:47+00:00,EURUSD.a,sell,OUT,1.25,1.17586,-90.25,-4.38,0.0,0.0,-94.63,[sl 1.17586]
59,2026-04-29 19:37:39+00:00,XAUUSD.a,sell,OUT,0.10,4545.83000,-93.84,0.00,0.0,0.0,-93.84,[sl 4545.83]
75,2026-04-30 14:27:52+00:00,XAUUSD.a,sell,OUT,0.10,4630.52000,-93.64,0.00,0.0,0.0,-93.64,[sl 4630.52]
13,2026-04-27 17:36:01+00:00,EURUSD.a,buy,OUT,1.25,1.17446,-83.37,-4.38,0.0,0.0,-87.75,[sl 1.17446]


## Position-level analysis

In [50]:
from Learn.report import (
    build_position_pairs,
    compute_trade_quality_metrics,
    equity_curve_and_drawdown,
    compute_rolling_metrics,
    compute_hourly_performance,
    compute_weekday_performance,
    compute_side_performance,
    compute_mae_mfe,
)

pairs = build_position_pairs(trades)
metrics = compute_trade_quality_metrics(pairs)

print(f"Completed positions : {metrics.get('trade_count', 0):,}")
print(f"Win rate            : {metrics.get('win_rate', 0):.1%}")
print(f"Profit factor       : {metrics.get('profit_factor', 0):.2f}")
print(f"Expectancy          : {metrics.get('expectancy', 0):.2f}")
print(f"Avg win             : {metrics.get('avg_win', 0):.2f}")
print(f"Avg loss            : {metrics.get('avg_loss', 0):.2f}")
print(f"Payoff ratio        : {metrics.get('payoff_ratio', 0):.2f}")
print(f"Total commission    : {metrics.get('total_commission', 0):.2f}")
print(f"Commission/trade    : {metrics.get('commission_per_trade', 0):.2f}")
print(f"Total net PnL       : {metrics.get('total_net_pnl', 0):.2f}")


Completed positions : 59
Win rate            : 57.6%
Profit factor       : 1.18
Expectancy          : 5.93
Avg win             : 68.04
Avg loss            : -78.54
Payoff ratio        : 0.87
Total commission    : -96.36
Commission/trade    : -1.63
Total net PnL       : 349.87


In [51]:

# Per-symbol metrics breakdown
for sym in sorted(pairs["symbol"].dropna().unique()):
    sym_pairs = pairs[pairs["symbol"] == sym]
    m = compute_trade_quality_metrics(sym_pairs)
    print(f"── {sym} ──────────────────────────────")
    print(f"  Completed positions : {m.get('trade_count', 0):,}")
    print(f"  Win rate            : {m.get('win_rate', 0):.1%}")
    print(f"  Profit factor       : {m.get('profit_factor', 0):.2f}")
    print(f"  Expectancy          : {m.get('expectancy', 0):.2f}")
    print(f"  Avg win             : {m.get('avg_win', 0):.2f}")
    print(f"  Avg loss            : {m.get('avg_loss', 0):.2f}")
    print(f"  Payoff ratio        : {m.get('payoff_ratio', 0):.2f}")
    print(f"  Total commission    : {m.get('total_commission', 0):.2f}")
    print(f"  Commission/trade    : {m.get('commission_per_trade', 0):.2f}")
    print(f"  Total net PnL       : {m.get('total_net_pnl', 0):.2f}")
    print()


── EURUSD.a ──────────────────────────────
  Completed positions : 11
  Win rate            : 36.4%
  Profit factor       : 0.35
  Expectancy          : -37.47
  Avg win             : 56.13
  Avg loss            : -90.96
  Payoff ratio        : 0.62
  Total commission    : -96.36
  Commission/trade    : -8.76
  Total net PnL       : -412.21

── US500.a ──────────────────────────────
  Completed positions : 10
  Win rate            : 80.0%
  Profit factor       : 6.01
  Expectancy          : 17.11
  Avg win             : 25.66
  Avg loss            : -17.09
  Payoff ratio        : 1.50
  Total commission    : 0.00
  Commission/trade    : 0.00
  Total net PnL       : 171.14

── XAUUSD.a ──────────────────────────────
  Completed positions : 38
  Win rate            : 57.9%
  Profit factor       : 1.46
  Expectancy          : 15.55
  Avg win             : 85.61
  Avg loss            : -80.78
  Payoff ratio        : 1.06
  Total commission    : 0.00
  Commission/trade    : 0.00
  Total net

## Equity curve and drawdown

In [52]:
if pairs.empty:
    print("No position pairs available for equity curve.")
else:
    curve, dd_metrics = equity_curve_and_drawdown(pairs)

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        row_heights=[0.65, 0.35],
        subplot_titles=("Equity Curve", "Drawdown"),
        vertical_spacing=0.06,
    )

    fig.add_trace(go.Scatter(
        x=curve["exit_time"], y=curve["equity"],
        mode="lines", name="Equity", line=dict(color="steelblue", width=2),
    ), row=1, col=1)

    # Horizontal zero line on equity chart
    fig.add_hline(y=0, line_dash="dot", line_color="grey", row=1, col=1)

    # Vertical line at max drawdown point
    min_dd_idx = curve["drawdown"].idxmin()
    fig.add_vline(
        x=curve.loc[min_dd_idx, "exit_time"],
        line_dash="dash", line_color="red", row=1, col=1,
    )

    fig.add_trace(go.Scatter(
        x=curve["exit_time"], y=curve["drawdown"],
        mode="lines", name="Drawdown", fill="tozeroy",
        line=dict(color="red", width=1),
        fillcolor="rgba(220,50,50,0.25)",
    ), row=2, col=1)

    fig.update_layout(
        title="Equity Curve & Drawdown",
        height=600, showlegend=False,
        xaxis2_title="Exit Time",
        yaxis_title="Cumulative PnL",
        yaxis2_title="Drawdown",
    )
    fig.show()

    print(f"Final equity          : {dd_metrics['final_equity']:.2f}")
    print(f"Max drawdown          : {dd_metrics['max_drawdown']:.2f}")
    print(f"Max drawdown %        : {dd_metrics['max_drawdown_pct']:.2f}%")
    print(f"Longest DD streak     : {dd_metrics['longest_drawdown_trades']} trades")
    print(f"Calmar ratio          : {dd_metrics['calmar_ratio']:.2f}")


Final equity          : 349.87
Max drawdown          : -468.08
Max drawdown %        : -130.12%
Longest DD streak     : 16 trades
Calmar ratio          : 0.75


## Buy vs sell signal performance

In [53]:
if pairs.empty:
    print("No position pairs available for side analysis.")
else:
    side_perf = compute_side_performance(pairs)
    display(side_perf)

    # Imbalanced buy/sell performance can indicate directional bias in the model;
    # e.g. consistently higher win-rate on buys in a bull market suggests the model
    # may need re-training or class-weight adjustment to improve sell-signal quality.

    sides = side_perf["side"].tolist()
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=sides, y=side_perf["trade_count"], name="Trade Count",
        marker_color="steelblue",
    ))
    fig.add_trace(go.Bar(
        x=sides, y=side_perf["win_rate"] * 100, name="Win Rate (%)",
        marker_color="mediumseagreen",
    ))
    fig.add_trace(go.Bar(
        x=sides, y=side_perf["avg_pnl"], name="Avg PnL",
        marker_color="darkorange",
    ))
    fig.update_layout(
        barmode="group",
        title="Buy vs Sell Performance",
        xaxis_title="Side",
        height=420,
    )
    fig.show()


,side,trade_count,win_rate,avg_pnl,total_pnl,avg_duration_mins
0,buy,36,0.472222,-9.955556,-358.40,16.314815
1,sell,23,0.739130,30.794348,708.27,12.060145


## Time-of-day performance (UTC)

In [54]:
if pairs.empty:
    print("No position pairs available for hourly analysis.")
else:
    hourly = compute_hourly_performance(pairs)

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        subplot_titles=("Trade Count by Hour", "Avg PnL by Hour"),
        vertical_spacing=0.1,
    )

    fig.add_trace(go.Bar(
        x=hourly["hour"], y=hourly["trade_count"],
        name="Trade Count", marker_color="steelblue",
    ), row=1, col=1)

    fig.add_trace(go.Bar(
        x=hourly["hour"], y=hourly["avg_pnl"],
        name="Avg PnL",
        marker_color=["green" if v >= 0 else "red" for v in hourly["avg_pnl"]],
    ), row=2, col=1)

    fig.update_layout(
        title="Time-of-Day Performance (UTC)",
        height=560, showlegend=False,
        xaxis2_title="Hour of Day (UTC)",
        yaxis_title="Trade Count",
        yaxis2_title="Avg PnL",
    )
    fig.show()


## Day-of-week performance

In [55]:
if pairs.empty:
    print("No position pairs available for weekday analysis.")
else:
    weekday = compute_weekday_performance(pairs)

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        subplot_titles=("Trade Count by Day", "Avg PnL by Day"),
        vertical_spacing=0.1,
    )

    fig.add_trace(go.Bar(
        x=weekday["weekday"], y=weekday["trade_count"],
        name="Trade Count", marker_color="steelblue",
    ), row=1, col=1)

    fig.add_trace(go.Bar(
        x=weekday["weekday"], y=weekday["avg_pnl"],
        name="Avg PnL",
        marker_color=["green" if v >= 0 else "red" for v in weekday["avg_pnl"]],
    ), row=2, col=1)

    fig.update_layout(
        title="Day-of-Week Performance",
        height=560, showlegend=False,
        xaxis2_title="Day of Week",
        yaxis_title="Trade Count",
        yaxis2_title="Avg PnL",
    )
    fig.show()


## Trade duration distribution

In [56]:
if pairs.empty:
    print("No position pairs available for duration analysis.")
else:
    import numpy as np
    p99 = pairs["duration_mins"].quantile(0.99)
    dur_df = pairs[pairs["duration_mins"] <= p99].copy()

    wins_dur = dur_df[dur_df["result"] == "win"]["duration_mins"]
    losses_dur = dur_df[dur_df["result"] == "loss"]["duration_mins"]

    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=("Duration Histogram (win vs loss)", "Duration Box Plot"),
        vertical_spacing=0.12,
    )

    fig.add_trace(go.Histogram(
        x=wins_dur, name="Win", opacity=0.65,
        marker_color="green", nbinsx=40,
    ), row=1, col=1)
    fig.add_trace(go.Histogram(
        x=losses_dur, name="Loss", opacity=0.65,
        marker_color="red", nbinsx=40,
    ), row=1, col=1)

    fig.add_trace(go.Box(
        y=wins_dur, name="Win", marker_color="green", boxmean=True,
    ), row=2, col=1)
    fig.add_trace(go.Box(
        y=losses_dur, name="Loss", marker_color="red", boxmean=True,
    ), row=2, col=1)

    fig.update_layout(
        barmode="overlay",
        title="Trade Duration Distribution",
        height=620,
        xaxis_title="Duration (mins)",
        yaxis_title="Count",
        yaxis2_title="Duration (mins)",
    )
    fig.show()

    med_win = wins_dur.median() if not wins_dur.empty else float('nan')
    med_loss = losses_dur.median() if not losses_dur.empty else float('nan')
    print(f"Median duration  wins : {med_win:.1f} mins")
    print(f"Median duration losses: {med_loss:.1f} mins")


Median duration  wins : 10.4 mins
Median duration losses: 7.4 mins


## Rolling win rate (last N trades)

In [57]:
ROLLING_WINDOW = 20  # configurable

if pairs.empty or len(pairs) < 2:
    print("Not enough trades for rolling metrics.")
else:
    rolling = compute_rolling_metrics(pairs, window=ROLLING_WINDOW)

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        subplot_titles=(
            f"Rolling Win Rate (window={ROLLING_WINDOW})",
            f"Rolling Avg PnL (window={ROLLING_WINDOW})",
        ),
        vertical_spacing=0.1,
    )

    fig.add_trace(go.Scatter(
        x=rolling["exit_time"], y=rolling["rolling_win_rate"] * 100,
        mode="lines", name="Win Rate %", line=dict(color="steelblue", width=2),
    ), row=1, col=1)
    fig.add_hline(y=50, line_dash="dash", line_color="grey", row=1, col=1)

    fig.add_trace(go.Scatter(
        x=rolling["exit_time"], y=rolling["rolling_avg_pnl"],
        mode="lines", name="Avg PnL", line=dict(color="darkorange", width=2),
    ), row=2, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color="grey", row=2, col=1)

    fig.update_layout(
        title=f"Rolling Performance (N={ROLLING_WINDOW})",
        height=560, showlegend=False,
        xaxis2_title="Exit Time",
        yaxis_title="Win Rate (%)",
        yaxis2_title="Avg PnL",
    )
    fig.show()


## Commission drag analysis

In [58]:
if pairs.empty:
    print("No position pairs available for commission analysis.")
else:
    gross_pnl = pairs["gross_pnl"].sum()
    total_commission = pairs["commission"].sum() + pairs["swap"].sum() + pairs["fee"].sum()
    net_pnl = pairs["net_pnl"].sum()
    commission_pct = abs(total_commission / gross_pnl * 100) if gross_pnl != 0 else float('nan')
    commission_per_trade = total_commission / len(pairs)

    print(f"Gross PnL            : {gross_pnl:.2f}")
    print(f"Total commission drag: {total_commission:.2f}")
    print(f"Net PnL              : {net_pnl:.2f}")
    print(f"Commission % of gross: {commission_pct:.2f}%")
    print(f"Commission per trade : {commission_per_trade:.2f}")

    # Waterfall chart
    fig = go.Figure(go.Waterfall(
        name="PnL Waterfall",
        orientation="v",
        measure=["relative", "relative", "total"],
        x=["Gross PnL", "Commission Drag", "Net PnL"],
        y=[gross_pnl, total_commission, 0],
        connector={"line": {"color": "rgb(63,63,63)"}},
        increasing={"marker": {"color": "green"}},
        decreasing={"marker": {"color": "red"}},
        totals={"marker": {"color": "steelblue"}},
    ))
    fig.update_layout(title="Commission Drag Waterfall", height=420, showlegend=False)
    fig.show()

    # Per-symbol breakdown if multiple symbols present
    if pairs["symbol"].nunique() > 1:
        sym_comm = (
            pairs.groupby("symbol")
            .agg(total_drag=("commission", lambda s: s.sum() + pairs.loc[s.index, "swap"].sum() + pairs.loc[s.index, "fee"].sum()))
            .reset_index()
        )
        fig2 = go.Figure(go.Bar(
            x=sym_comm["symbol"], y=sym_comm["total_drag"],
            marker_color=["green" if v >= 0 else "red" for v in sym_comm["total_drag"]],
        ))
        fig2.update_layout(
            title="Commission Drag by Symbol",
            xaxis_title="Symbol", yaxis_title="Total Drag",
            height=380,
        )
        fig2.show()


Gross PnL            : 446.23
Total commission drag: -96.36
Net PnL              : 349.87
Commission % of gross: 21.59%
Commission per trade : -1.63


## Maximum adverse / favorable excursion (MAE / MFE)

In [59]:
import numpy as np

MAE_MFE_SYMBOL = SYMBOL  # uses top-level config

if ohlcv is None or ohlcv.empty:
    print("No OHLCV data available for MAE/MFE analysis.")
elif pairs.empty:
    print("No position pairs available for MAE/MFE analysis.")
else:
    mae_mfe = compute_mae_mfe(pairs, ohlcv, symbol=MAE_MFE_SYMBOL)

    if mae_mfe.empty:
        print("MAE/MFE computation returned no results (check symbol filter or OHLCV date range).")
    else:
        wins_mm = mae_mfe[mae_mfe["result"] == "win"]
        losses_mm = mae_mfe[mae_mfe["result"] == "loss"]

        color_map = {"win": "green", "loss": "red", "breakeven": "grey"}
        colors = [color_map.get(r, "grey") for r in mae_mfe["result"]]

        fig = make_subplots(
            rows=2, cols=1,
            subplot_titles=("MFE vs MAE Scatter", "MAE Distribution (wins vs losses)"),
            vertical_spacing=0.12,
        )

        fig.add_trace(go.Scatter(
            x=mae_mfe["mae_pts"], y=mae_mfe["mfe_pts"],
            mode="markers",
            marker=dict(color=colors, size=7, opacity=0.75, line=dict(width=0.5, color="white")),
            name="Trades",
            text=mae_mfe["result"],
        ), row=1, col=1)

        # Diagonal reference line: MFE == |MAE| (i.e. MFE = -MAE)
        min_mae = float(mae_mfe["mae_pts"].min())
        fig.add_trace(go.Scatter(
            x=[min_mae, 0], y=[-min_mae, 0],
            mode="lines", name="MFE=|MAE|",
            line=dict(color="grey", dash="dash", width=1),
        ), row=1, col=1)

        fig.add_trace(go.Histogram(
            x=wins_mm["mae_pts"], name="Win MAE",
            marker_color="green", opacity=0.65, nbinsx=40,
        ), row=2, col=1)
        fig.add_trace(go.Histogram(
            x=losses_mm["mae_pts"], name="Loss MAE",
            marker_color="red", opacity=0.65, nbinsx=40,
        ), row=2, col=1)

        fig.update_layout(
            barmode="overlay",
            title=f"MAE / MFE Analysis — {MAE_MFE_SYMBOL}",
            height=700,
            xaxis_title="MAE (price pts)",
            yaxis_title="MFE (price pts)",
            xaxis2_title="MAE (price pts)",
            yaxis2_title="Count",
        )
        fig.show()

        # Interpretation: trades where MAE is large relative to MFE suggest poor entry
        # timing or TP/SL miscalibration — the position moved heavily against before any
        # favourable move, implying entries are late or stops are too wide.

        avg_mae_win = wins_mm["mae_pts"].mean() if not wins_mm.empty else float('nan')
        avg_mae_loss = losses_mm["mae_pts"].mean() if not losses_mm.empty else float('nan')
        avg_mfe_win = wins_mm["mfe_pts"].mean() if not wins_mm.empty else float('nan')
        med_ratio = mae_mfe["mfe_to_mae_ratio"].median()

        print(f"Avg MAE  (wins)      : {avg_mae_win:.4f} pts")
        print(f"Avg MAE  (losses)    : {avg_mae_loss:.4f} pts")
        print(f"Avg MFE  (wins)      : {avg_mfe_win:.4f} pts")
        print(f"Median MFE/MAE ratio : {med_ratio:.2f}")


Avg MAE  (wins)      : 6.5736 pts
Avg MAE  (losses)    : -8.2960 pts
Avg MFE  (wins)      : 14.0932 pts
Median MFE/MAE ratio : 1.10
